# Compare to NL/BE/DE network

- looking to compare our network for the UK to the existing network across Western Europe 
- try to download full node network
- download street network to check that we're controlling for the density of streets (and street type)

## 1. Setup

In [ ]:
# libaries
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
import requests
import os
import pandas as pd
import time

In [ ]:
belgian_arrondissements = gpd.read_file(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\cycleNodes\data\boundaries\georef-belgium-arrondissement-millesime.shp")
belgian_arrondissements.plot()

In [ ]:
dutch_COROP = gpd.read_file(r"C:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\cycleNodes\data\boundaries\cbsgebiedsindelingen2025.gpkg")
dutch_COROP.plot()

## 2. Get node networks

In [6]:
# # Download bike network
# G = ox.graph_from_place(place, network_type="bike")
# nodes, edges = ox.graph_to_gdfs(G)


# #  Get place polygon
# poly = ox.geocoder.geocode_to_gdf(place).geometry.unary_union
# gdf_poly = ox.geocoder.geocode_to_gdf(place)

# # convert polygon to Overpass poly string (lat lon order)
# poly_simple = poly.simplify(0.001)  # ~100 m tolerance
# poly_coords = " ".join([f"{y} {x}" for x, y in poly_simple.exterior.coords])


# # Query Overpass for node network relations only and their members
# query = f"""
# [out:json][timeout:180];
# (
#   relation["route"="bicycle"]["network"="rcn"]["network:type"="node_network"](poly:"{poly_coords}");
# );
# (._;>;);
# out;
# """

# url = "https://overpass-api.de/api/interpreter"
# response = requests.post(url, data=query)
# data = response.json()


# # Extract way IDs that belong to RCN routes
# rcn_way_ids = {
#     el["id"]
#     for el in data["elements"]
#     if el["type"] == "way"
# }

# print("RCN ways found:", len(rcn_way_ids))


# # Match these ways to the OSMnx edges
# def edge_is_rcn(osmid):
#     if isinstance(osmid, list):
#         return any(w in rcn_way_ids for w in osmid)
#     return osmid in rcn_way_ids

# edges["rcn"] = edges["osmid"].apply(edge_is_rcn)

# rcn_edges = edges[edges["rcn"]].copy()

# print("RCN edges matched:", len(rcn_edges))

# # Get RCN junction nodes (knooppunten)
# tags = {"rcn_ref": True}
# rcn_nodes = ox.features_from_place(place, tags)

# print("RCN nodes:", len(rcn_nodes))


# ax = rcn_edges.plot(figsize=(8,8), linewidth=1)
# rcn_nodes.plot(ax=ax, color="red", markersize=10)
# gdf_poly.boundary.plot(ax=ax, color="black", linewidth=1)
# plt.title(f"fietsknoop Bicycle Network in {place} from OpenStreetMap")

---------------------

In [7]:
# place = "Tongeren, Belgium"

# # Download bike network
# G = ox.graph_from_place(place, network_type="bike")
# nodes, edges = ox.graph_to_gdfs(G)

# # Get place polygon
# gdf_poly = ox.geocoder.geocode_to_gdf(place)
# poly = gdf_poly.geometry.unary_union

# # Get the bounding box of the polygon for the Overpass query (south, west, north, east)
# minx, miny, maxx, maxy = gdf_poly.total_bounds

# # Query Overpass for node network relations using bounding box instead of poly string
# query = f"""
# [out:json][timeout:180];
# (
#   relation["route"="bicycle"]["network"="rcn"]["network:type"="node_network"]({miny},{minx},{maxy},{maxx});
# );
# (._;>;);
# out;
# """

# url = "https://overpass-api.de/api/interpreter"
# # Send query explicitly as the 'data' parameter payload
# response = requests.post(url, data={'data': query})

# if response.status_code != 200:
#     print(f"Server Error {response.status_code}:")
#     print(response.text[:500])  # Print the actual text error from Overpass
#     data = {"elements": []} # Fallback to prevent crash
# else:
#     data = response.json()

# # Extract way IDs that belong to RCN routes
# rcn_way_ids = {
#     el["id"]
#     for el in data.get("elements", [])
#     if el["type"] == "way"
# }

# print("RCN ways found:", len(rcn_way_ids))

# # Match these ways to the OSMnx edges
# def edge_is_rcn(osmid):
#     if isinstance(osmid, list):
#         return any(w in rcn_way_ids for w in osmid)
#     return osmid in rcn_way_ids

# edges["rcn"] = edges["osmid"].apply(edge_is_rcn)

# rcn_edges = edges[edges["rcn"]].copy()

# # Get RCN junction nodes (knooppunten)
# tags = {"rcn_ref": True}
# rcn_nodes = ox.features_from_place(place, tags)

# # --- Clip edges and nodes precisely down to the actual boundary ---
# if not rcn_edges.empty:
#     if rcn_edges.crs != gdf_poly.crs:
#         rcn_edges = rcn_edges.to_crs(gdf_poly.crs)
#     rcn_edges = gpd.clip(rcn_edges, gdf_poly)

# if not rcn_nodes.empty:
#     if rcn_nodes.crs != gdf_poly.crs:
#         rcn_nodes = rcn_nodes.to_crs(gdf_poly.crs)
#     rcn_nodes = gpd.clip(rcn_nodes, gdf_poly)

# print("RCN edges strictly matched inside boundary:", len(rcn_edges))
# print("RCN nodes strictly matched inside boundary:", len(rcn_nodes))

# # Plot
# ax = rcn_edges.plot(figsize=(8,8), linewidth=1)
# if not rcn_nodes.empty:
#     rcn_nodes.plot(ax=ax, color="red", markersize=10)
# gdf_poly.boundary.plot(ax=ax, color="black", linewidth=1)
# plt.title(f"fietsknoop Bicycle Network in {place} from OpenStreetMap")

# Analyse existing networks

In [8]:
# # Compute area (km^2), total RCN length (km), node count and density (km / km^2)
# poly_m = gdf_poly.to_crs(epsg=3857).geometry.iloc[0]          # metric CRS
# area_km2 = poly_m.area / 1e6

# rcn_edges_m = rcn_edges.to_crs(epsg=3857)
# total_length_km = rcn_edges_m.geometry.length.sum() / 1000

# n_rcn_nodes = len(rcn_nodes)

# density_km_per_km2 = total_length_km / area_km2 if area_km2 > 0 else float('nan')

# print(f"Place: {place}")
# print(f"Area: {area_km2:.3f} km^2")
# print(f"RCN total length: {total_length_km:.3f} km")
# print(f"RCN nodes: {n_rcn_nodes}")
# print(f"RCN length density: {density_km_per_km2:.3f} km / km^2")


In [21]:

def analyze_networks_from_gdf(gdf, name_col, output_dir="data"):
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    results = []
    
    # OSMnx networks require unprojected WGS84 (lat/lon) geometries
    if gdf.crs != "EPSG:4326":
        gdf = gdf.to_crs("EPSG:4326")
        
    for idx, row in gdf.iterrows():
        place_name = str(getattr(row, name_col))
        print(f"Processing: {place_name}...")
        
        try:
            time.sleep(5)  # Shorter sleep to avoid rapid rate limits
            
            poly = row.geometry
            if not poly or poly.is_empty:
                print(f"  Invalid geometry for {place_name}, skipping.")
                continue

            # 1. Download bike network using the geometry directly
            try:
                G = ox.graph_from_polygon(poly, network_type="bike")
                nodes, edges = ox.graph_to_gdfs(G)
            except Exception as e:
                print(f"  Could not get OSMnx graph for {place_name}: {e}")
                continue

            # 2. Get bounds of the polygon for Overpass query
            minx, miny, maxx, maxy = poly.bounds

            # 3. Query Overpass for node network relations via bounding box
            query = f"""
            [out:json][timeout:180];
            (
              relation["route"="bicycle"]["network"="rcn"]["network:type"="node_network"]({miny},{minx},{maxy},{maxx});
            );
            (._;>;);
            out;
            """
            
            url = "https://overpass-api.de/api/interpreter"
            max_retries = 5
            for attempt in range(max_retries):
                response = requests.post(url, data={'data': query})
                
                if response.status_code == 200:
                    data = response.json()
                    break # Success, break out of retry loop
                else:
                    print(f"  Overpass Error {response.status_code} on attempt {attempt + 1}. Waiting 120s...")
                    time.sleep(120) # Wait 120 seconds before retrying
                    data = {"elements": []}

            # Extract way IDs that belong to RCN routes
            rcn_way_ids = {el["id"] for el in data.get("elements", []) if el["type"] == "way"}
            
            # Match these ways to the OSMnx edges
            def edge_is_rcn(osmid):
                if isinstance(osmid, list):
                    return any(w in rcn_way_ids for w in osmid)
                return osmid in rcn_way_ids

            edges["rcn"] = edges["osmid"].apply(edge_is_rcn)
            rcn_edges = edges[edges["rcn"]].copy()

            # 4. Get RCN junction nodes using the geometry
            tags = {"rcn_ref": True}
            try:
                rcn_nodes = ox.features_from_polygon(poly, tags)
            except Exception:
                rcn_nodes = None

            # 5. Create a temporary GeoDataFrame of just this polygon for plotting and clipping
            gdf_poly = gpd.GeoDataFrame({"geometry": [poly]}, crs="EPSG:4326")

            # 6. Clip strictly to the polygon boundary
            if not rcn_edges.empty:
                if rcn_edges.crs != gdf_poly.crs:
                    rcn_edges = rcn_edges.to_crs(gdf_poly.crs)
                rcn_edges = gpd.clip(rcn_edges, gdf_poly)
            
            if rcn_nodes is not None and not rcn_nodes.empty:
                if rcn_nodes.crs != gdf_poly.crs:
                    rcn_nodes = rcn_nodes.to_crs(gdf_poly.crs)
                rcn_nodes = gpd.clip(rcn_nodes, gdf_poly)
                n_rcn_nodes = len(rcn_nodes)
            else:
                n_rcn_nodes = 0
                rcn_nodes = gpd.GeoDataFrame()

            # 7. Plot and save
            fig, ax = plt.subplots(figsize=(8,8))
            if not rcn_edges.empty:
                rcn_edges.plot(ax=ax, linewidth=1, color="blue", label="RCN Edges")
            if not rcn_nodes.empty:
                rcn_nodes.plot(ax=ax, color="red", markersize=10, zorder=5, label="Nodes")
            
            gdf_poly.boundary.plot(ax=ax, color="black", linewidth=1)
            plt.title(f"fietsknoop Bicycle Network in {place_name}")
            
            clean_name = place_name.replace(", ", "_").replace(" ", "_").replace("/", "-")
            plot_path = os.path.join(output_dir, f"{str(clean_name)}_network.png")
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.close(fig) 

            # 8. Compute Stats
            poly_m = gdf_poly.to_crs(epsg=3857).geometry.iloc[0]
            area_km2 = poly_m.area / 1e6

            if not rcn_edges.empty:
                rcn_edges_m = rcn_edges.to_crs(epsg=3857)
                total_length_km = rcn_edges_m.geometry.length.sum() / 1000
            else:
                total_length_km = 0.0

            density_km_per_km2 = total_length_km / area_km2 if area_km2 > 0 else float('nan')
            
            # Print stats
            print(f"  Area: {area_km2:.3f} km^2")
            print(f"  RCN total length: {total_length_km:.3f} km")
            print(f"  RCN nodes: {n_rcn_nodes}")
            print(f"  RCN length density: {density_km_per_km2:.3f} km / km^2\n")

            # Append to results
            results.append({
                "Place": place_name,
                "Area_km2": round(area_km2, 3),
                "RCN_Total_Length_km": round(total_length_km, 3),
                "RCN_Nodes": n_rcn_nodes,
                "RCN_Length_Density_km_per_km2": round(density_km_per_km2, 3) 
            })

        except Exception as e:
            print(f"  Error processing {place_name}: {e}\n")

    # Save all results to CSV
    if results:
        results_df = pd.DataFrame(results)
        csv_path = os.path.join(output_dir, f"european_network_stats_{int(time.time())}.csv")
        results_df.to_csv(csv_path, index=False)
        print(f"Successfully saved aggregated statistics to: {csv_path}")
        return results_df
    else:
        print("No successful results to save.")
        return None

# To run this on the geometries you loaded, you will call it like this:
# (Make sure to replace 'name_column' with the actual column inside your shapefile containing the name)
belgian_stats_df = analyze_networks_from_gdf(belgian_arrondissements, name_col="Arrondissem")
display(belgian_stats_df)
dutch_stats_df = analyze_networks_from_gdf(dutch_COROP, name_col="statnaam")

Processing: BEZIRK LÜTTICH...
  Area: 1974.380 km^2
  RCN total length: 1027.044 km
  RCN nodes: 107
  RCN length density: 0.520 km / km^2

Processing: BEZIRK SOIGNIES...
  Area: 885.224 km^2
  RCN total length: 1021.105 km
  RCN nodes: 113
  RCN length density: 1.153 km / km^2

Processing: BEZIRK WAREMME...
  Overpass Error 429 on attempt 1. Waiting 30s...
  Area: 970.474 km^2
  RCN total length: 545.943 km
  RCN nodes: 62
  RCN length density: 0.563 km / km^2

Processing: BEZIRK DENDERMONDE...
  Area: 875.364 km^2
  RCN total length: 1280.775 km
  RCN nodes: 124
  RCN length density: 1.463 km / km^2

Processing: BEZIRK ARLON...
  Area: 760.682 km^2
  RCN total length: 491.233 km
  RCN nodes: 41
  RCN length density: 0.646 km / km^2

Processing: BEZIRK NAMUR...
  Area: 2878.075 km^2
  RCN total length: 2202.500 km
  RCN nodes: 164
  RCN length density: 0.765 km / km^2

Processing: BEZIRK ATH...
  Area: 1668.463 km^2
  RCN total length: 2257.791 km
  RCN nodes: 331
  RCN length density

,Place,Area_km2,RCN_Total_Length_km,RCN_Nodes,RCN_Length_Density_km_per_km2
0,BEZIRK LÜTTICH,1974.380,1027.044,107,0.520
1,BEZIRK SOIGNIES,885.224,1021.105,113,1.153
2,BEZIRK WAREMME,970.474,545.943,62,0.563
3,BEZIRK DENDERMONDE,875.364,1280.775,124,1.463
4,BEZIRK ARLON,760.682,491.233,41,0.646
5,BEZIRK NAMUR,2878.075,2202.500,164,0.765
6,BEZIRK ATH,1668.463,2257.791,331,1.353
7,BEZIRK HASSELT,2309.345,2246.582,170,0.973
8,BEZIRK VIRTON,1853.240,1344.426,111,0.725
9,BEZIRK ANTWERPEN,2511.365,3148.879,322,1.254


Processing: Groningen...
  Area: 8621.598 km^2
  RCN total length: 9275.083 km
  RCN nodes: 818
  RCN length density: 1.076 km / km^2

Processing: Friesland...
  Area: 9796.902 km^2
  RCN total length: 10000.726 km
  RCN nodes: 945
  RCN length density: 1.021 km / km^2

Processing: Drenthe...
  Overpass Error 504 on attempt 1. Waiting 30s...
  Overpass Error 504 on attempt 2. Waiting 30s...
  Area: 4436.295 km^2
  RCN total length: 4492.521 km
  RCN nodes: 331
  RCN length density: 1.013 km / km^2

Processing: Twente...
  Area: 4017.315 km^2
  RCN total length: 5964.538 km
  RCN nodes: 527
  RCN length density: 1.485 km / km^2

Processing: Veluwe Stedendriehoek...
  Overpass Error 504 on attempt 1. Waiting 30s...
  Overpass Error 504 on attempt 2. Waiting 30s...
  Area: 4386.064 km^2
  RCN total length: 5151.907 km
  RCN nodes: 557
  RCN length density: 1.175 km / km^2

Processing: Midden-Gelderland...
  Overpass Error 504 on attempt 1. Waiting 30s...
  Area: 1473.122 km^2
  RCN total 

In [ ]:
## OPENSTREETMAP ONLY

def analyze_european_networks(places, output_dir="data"):
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    results = []
    for place in places:
        print(f"Processing: {place}...")
        try:
            time.sleep(120)  # Sleep to avoid hitting Overpass rate limits

            #  Download bike network
            G = ox.graph_from_place(place, network_type="bike")
            nodes, edges = ox.graph_to_gdfs(G)

            #  Get place polygon
            gdf_poly = ox.geocoder.geocode_to_gdf(place)
            poly = gdf_poly.geometry.unary_union

            #  Convert polygon to Overpass poly string (lat lon order)
            poly_simple = poly.simplify(0.001)  # ~100 m tolerance
            poly_coords = " ".join([f"{y} {x}" for x, y in poly_simple.exterior.coords])

            #  Query Overpass for node network relations
            query = f"""
            [out:json][timeout:180];
            (
              relation["route"="bicycle"]["network"="rcn"]["network:type"="node_network"](poly:"{poly_coords}");
            );
            out body;
            >;
            out skel qt;
            """
            
            url = "https://overpass-api.de/api/interpreter"
            response = requests.post(url, data=query)
            data = response.json()

            # Extract way IDs that belong to RCN routes
            rcn_way_ids = {el["id"] for el in data["elements"] if el["type"] == "way"}
            
            # Match these ways to the OSMnx edges
            def edge_is_rcn(osmid):
                if isinstance(osmid, list):
                    return any(w in rcn_way_ids for w in osmid)
                return osmid in rcn_way_ids

            edges["rcn"] = edges["osmid"].apply(edge_is_rcn)
            rcn_edges = edges[edges["rcn"]].copy()

            # Get RCN junction nodes
            tags = {"rcn_ref": True}
            try:
                rcn_nodes = ox.features_from_place(place, tags)
                n_rcn_nodes = len(rcn_nodes)
            except Exception:
                # Handle cases where no nodes are found
                n_rcn_nodes = 0
                rcn_nodes = None

            # 6. Plot and save
            fig, ax = plt.subplots(figsize=(8,8))
            if not rcn_edges.empty:
                rcn_edges.plot(ax=ax, linewidth=1, color="blue", label="RCN Edges")
            if rcn_nodes is not None and not rcn_nodes.empty:
                rcn_nodes.plot(ax=ax, color="red", markersize=10, label="Nodes")
            
            gdf_poly.boundary.plot(ax=ax, color="black", linewidth=1)
            plt.title(f"fietsknoop Bicycle Network in {place}")
            
            # Save the plot
            clean_name = place.replace(", ", "_").replace(" ", "_")
            plot_path = os.path.join(output_dir, f"{clean_name}_network.png")
            plt.savefig(plot_path, dpi=300, bbox_inches='tight')
            plt.close(fig) 

            # Compute Stats
            poly_m = gdf_poly.to_crs(epsg=3857).geometry.iloc[0]
            area_km2 = poly_m.area / 1e6

            if not rcn_edges.empty:
                rcn_edges_m = rcn_edges.to_crs(epsg=3857)
                total_length_km = rcn_edges_m.geometry.length.sum() / 1000
            else:
                total_length_km = 0.0

            density_km_per_km2 = total_length_km / area_km2 if area_km2 > 0 else float('nan')
            
            # Print stats
            print(f"  Area: {area_km2:.3f} km^2")
            print(f"  RCN total length: {total_length_km:.3f} km")
            print(f"  RCN nodes: {n_rcn_nodes}")
            print(f"  RCN length density: {density_km_per_km2:.3f} km / km^2")
            print(f"  Saved plot to: {plot_path}\n")

            # Append to results
            results.append({
                "Place": place,
                "Area_km2": round(area_km2, 3),
                "RCN_Total_Length_km": round(total_length_km, 3),
                "RCN_Nodes": n_rcn_nodes,
                "RCN_Length_Density_km_per_km2": round(density_km_per_km2, 3) })

        except Exception as e:
            print(f"  Error processing {place}: {e}\n")

    # Save all results to CSV
    if results:
        results_df = pd.DataFrame(results)
        csv_path = os.path.join(output_dir, "european_network_stats.csv")
        results_df.to_csv(csv_path, index=False)
        print(f"Successfully saved aggregated statistics to: {csv_path}")
        return results_df
    else:
        print("No successful results to save.")
        return None

# Run the function 
places_to_analyze = [
    "Statenkieskring 1 (Maastricht)",
    "Tongeren, Limburg, Flanders, Belgium",
    "Maaseik, Limburg, Flanders, Belgium",
    "Metropolitan Region Eindhoven, North Brabant, Netherlands",
    "Drenthe, Netherlands",
    "Noord-Holland Noord, Netherlands",
]

stats_df = analyze_european_networks(places_to_analyze)
display(stats_df)

Processing: Statenkieskring 1 (Maastricht)...
  Error processing Statenkieskring 1 (Maastricht): Expecting value: line 1 column 1 (char 0)

Processing: Tongeren, Limburg, Flanders, Belgium...


c:\Users\b8008458\AppData\Local\miniforge3\envs\growbikenet\Lib\site-packages\IPython\core\pylabtools.py:77: DeprecationWarning: backend2gui is deprecated since IPython 8.24, backends are managed in matplotlib and can be externally registered.
  warnings.warn(


  Area: 32.985 km^2
  RCN total length: 50.650 km
  RCN nodes: 7
  RCN length density: 1.536 km / km^2
  Saved plot to: data\Tongeren_Limburg_Flanders_Belgium_network.png

Processing: Maaseik, Limburg, Flanders, Belgium...
  Area: 195.359 km^2
  RCN total length: 205.957 km
  RCN nodes: 10
  RCN length density: 1.054 km / km^2
  Saved plot to: data\Maaseik_Limburg_Flanders_Belgium_network.png

Processing: Metropolitan Region Eindhoven, North Brabant, Netherlands...
  Area: 3742.607 km^2
  RCN total length: 4508.728 km
  RCN nodes: 543
  RCN length density: 1.205 km / km^2
  Saved plot to: data\Metropolitan_Region_Eindhoven_North_Brabant_Netherlands_network.png

Processing: Drenthe, Netherlands...
  Area: 7340.726 km^2
  RCN total length: 8014.407 km
  RCN nodes: 626
  RCN length density: 1.092 km / km^2
  Saved plot to: data\Drenthe_Netherlands_network.png

Processing: Noord-Holland Noord, Netherlands...
  Area: 6909.432 km^2
  RCN total length: 3953.979 km
  RCN nodes: 403
  RCN lengt

,Place,Area_km2,RCN_Total_Length_km,RCN_Nodes,RCN_Length_Density_km_per_km2
0,"Tongeren, Limburg, Flanders, Belgium",32.985,50.650,7,1.536
1,"Maaseik, Limburg, Flanders, Belgium",195.359,205.957,10,1.054
2,"Metropolitan Region Eindhoven, North Brabant, ...",3742.607,4508.728,543,1.205
3,"Drenthe, Netherlands",7340.726,8014.407,626,1.092
4,"Noord-Holland Noord, Netherlands",6909.432,3953.979,403,0.572


# Compare to our areas

## Load

In [10]:
place = "Gateshead, United Kingdom" 
current_wd = os.getcwd()

# Construct the path 
gpkg_path = os.path.join(current_wd, "Cycle-node-network-feasibility", "data", f"{place.replace(', ', '_')}_network.gpkg")

# Read the layers
boundary_gdf = gpd.read_file(gpkg_path, layer='boundary')
nodes_gdf = gpd.read_file(gpkg_path, layer='processed_nodes')
constrained_network_gdf = gpd.read_file(gpkg_path, layer='constrained_routes')
relaxed_constrained_network_gdf = gpd.read_file(gpkg_path, layer='relaxed_constrained_routes')
fully_connected_network_gdf = gpd.read_file(gpkg_path, layer='fully_connected_routes')


# Verify what was loaded
print(f"Loaded boundary: {len(boundary_gdf)} polygons")
print(f"Loaded nodes: {len(nodes_gdf)} points")
print(f"Loaded constrained network: {len(constrained_network_gdf)} edges")
print(f"Loaded relaxed constrained network: {len(relaxed_constrained_network_gdf)} edges")
print(f"Loaded fully connected network: {len(fully_connected_network_gdf)} edges")

DriverError: Failed to open dataset (flags=68): c:\Users\b8008458\OneDrive - Newcastle University\2022 to 2023\PhD\cycleNodes\Cycle-node-network-feasibility\Cycle-node-network-feasibility\data\Gateshead_United Kingdom_network.gpkg

## Analyse

In [ ]:
# Compute area using a metric CRS (EPSG:3857)
boundary_m = boundary_gdf.to_crs(epsg=3857).geometry.iloc[0]
area_km2 = boundary_m.area / 1e6

n_nodes_uk = len(nodes_gdf)

print(f"Place: {place}")
print(f"Area: {area_km2:.3f} km^2")
print(f"Generated seed nodes: {n_nodes_uk}")
print("-" * 30)

# Dictionary of the networks to analyze
networks = {
    "Constrained": constrained_network_gdf,
    "Relaxed Constrained": relaxed_constrained_network_gdf,
    "Fully Connected": fully_connected_network_gdf
}

# Compute length and density for each variant
for name, net_gdf in networks.items():
    if len(net_gdf) > 0:
        # Project to metric CRS to get lengths in meters
        net_m = net_gdf.to_crs(epsg=3857)
        total_length_km = net_m.geometry.length.sum() / 1000
        density_km_per_km2 = total_length_km / area_km2 if area_km2 > 0 else float('nan')
        
        print(f"--- {name} Network ---")
        print(f"Total length: {total_length_km:.3f} km")
        print(f"Length density: {density_km_per_km2:.3f} km / km^2")
    else:
        print(f"--- {name} Network ---")
        print("No edges found in this network.")

Place: Gateshead, United Kingdom
Area: 435.863 km^2
Generated seed nodes: 59
------------------------------
--- Constrained Network ---
Total length: 173.405 km
Length density: 0.398 km / km^2
--- Relaxed Constrained Network ---
Total length: 351.072 km
Length density: 0.805 km / km^2
--- Fully Connected Network ---
Total length: 869.073 km
Length density: 1.994 km / km^2


In [ ]:
# Compute area using a metric CRS (EPSG:3857)
boundary_m = boundary_gdf.to_crs(epsg=3857).geometry.iloc[0]
area_km2 = boundary_m.area / 1e6

n_nodes_uk = len(nodes_gdf)

print(f"Place: {place}")
print(f"Area: {area_km2:.3f} km^2")
print(f"Generated seed nodes: {n_nodes_uk}")
print("-" * 30)

# Dictionary of the networks to analyze
networks = {
    "Constrained": constrained_network_gdf,
    "Relaxed Constrained": relaxed_constrained_network_gdf,
    "Fully Connected": fully_connected_network_gdf
}

results_uk = []
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

# Compute length and density, save plots, and build stats for each variant
for name, net_gdf in networks.items():
    if len(net_gdf) > 0:
        # Project to metric CRS to get lengths in meters
        net_m = net_gdf.to_crs(epsg=3857)
        total_length_km = net_m.geometry.length.sum() / 1000
        density_km_per_km2 = total_length_km / area_km2 if area_km2 > 0 else float('nan')
        
        print(f"--- {name} Network ---")
        print(f"Total length: {total_length_km:.3f} km")
        print(f"Length density: {density_km_per_km2:.3f} km / km^2")
        
        # Plot and save
        fig, ax = plt.subplots(figsize=(8,8))
        net_gdf.plot(ax=ax, linewidth=1, color="blue", label="Edges")
        nodes_gdf.plot(ax=ax, color="red", markersize=10, zorder=5, label="Nodes")
        boundary_gdf.boundary.plot(ax=ax, color="black", linewidth=1)
        plt.title(f"{name} Simulated Network in {place}")
        
        clean_name = place.replace(", ", "_").replace(" ", "_")
        plot_path = os.path.join(output_dir, f"{clean_name}_{name.replace(' ','_')}_network.png")
        plt.savefig(plot_path, dpi=300, bbox_inches='tight')
        plt.close(fig) 
        
        # Append to results
        results_uk.append({
            "Place": f"{place} ({name})",
            "Area_km2": round(area_km2, 3),
            "RCN_Total_Length_km": round(total_length_km, 3),
            "RCN_Nodes": n_nodes_uk,
            "RCN_Length_Density_km_per_km2": round(density_km_per_km2, 3) 
        })
    else:
        print(f"--- {name} Network ---")
        print("No edges found in this network.")

# Create DataFrame and print
gateshead_stats_df = pd.DataFrame(results_uk)
display(gateshead_stats_df)